# E7 — Transfer Learning VGG16 + Oversampling + Data Augmentation

E7 reprend le protocole de E6 et ajoute uniquement un oversampling du jeu d'entraînement.

**Hypothèse provisoire à confirmer avec le binôme :** `RandomOverSampler(random_state=42)` équilibre la classe COVID sur la classe majoritaire (50/50).  
La validation et le test restent inchangés. Le test n'est pas utilisé pour sélectionner le modèle.


In [ ]:
import pandas as pd

# Split figé commun à toutes les expériences
train_df = pd.read_csv("../covid/splits/train.csv")
val_df = pd.read_csv("../covid/splits/validation.csv")
test_df = pd.read_csv("../covid/splits/test.csv")

print("Train original :")
print(train_df["target"].value_counts())
print(train_df["target"].value_counts(normalize=True))


In [ ]:
from imblearn.over_sampling import RandomOverSampler

# Oversampling uniquement sur le train.
# Méthode alignée sur la dernière version disponible du notebook du binôme.
# À reconfirmer avec lui avant le lancement définitif de E7.
X_train_e7 = train_df.drop(columns=["target"])
y_train_e7 = train_df["target"]

ros_e7 = RandomOverSampler(random_state=42)
X_train_e7_os, y_train_e7_os = ros_e7.fit_resample(X_train_e7, y_train_e7)

train_e7_df = X_train_e7_os.copy()
train_e7_df["target"] = y_train_e7_os.values

# Mélange reproductible après oversampling
train_e7_df = train_e7_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Train après oversampling :")
print(train_e7_df["target"].value_counts())
print(train_e7_df["target"].value_counts(normalize=True))
print("Shape :", train_e7_df.shape)

# Contrôles : validation/test non modifiés
print("\nValidation inchangée :")
print(val_df["target"].value_counts())
print("\nTest inchangé :")
print(test_df["target"].value_counts())


In [ ]:
# Copies dédiées aux générateurs Keras
train_gen_df_e7 = train_e7_df.copy()
val_gen_df_e7 = val_df.copy()
test_gen_df_e7 = test_df.copy()

train_gen_df_e7["target"] = train_gen_df_e7["target"].astype(str)
val_gen_df_e7["target"] = val_gen_df_e7["target"].astype(str)
test_gen_df_e7["target"] = test_gen_df_e7["target"].astype(str)


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input

# Même Data Augmentation que E6
train_datagen_e7 = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)

# Validation : preprocessing VGG16 uniquement
val_datagen_e7 = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)
COLOR_MODE = "rgb"

train_generator_e7 = train_datagen_e7.flow_from_dataframe(
    dataframe=train_gen_df_e7,
    x_col="filepath",
    y_col="target",
    target_size=IMAGE_SIZE,
    color_mode=COLOR_MODE,
    class_mode="binary",
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

val_generator_e7 = val_datagen_e7.flow_from_dataframe(
    dataframe=val_gen_df_e7,
    x_col="filepath",
    y_col="target",
    target_size=IMAGE_SIZE,
    color_mode=COLOR_MODE,
    class_mode="binary",
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Préparé pour l'évaluation finale uniquement : ne pas utiliser pour sélectionner E7.
test_generator_e7 = val_datagen_e7.flow_from_dataframe(
    dataframe=test_gen_df_e7,
    x_col="filepath",
    y_col="target",
    target_size=IMAGE_SIZE,
    color_mode=COLOR_MODE,
    class_mode="binary",
    batch_size=BATCH_SIZE,
    shuffle=False
)


In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout

# Même architecture que E4/E6 : nouveau modèle indépendant
base_model_e7 = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model_e7.trainable = False

model_e7 = Sequential([
    base_model_e7,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

model_e7.summary()


In [ ]:
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model_e7.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

early_stopping_e7 = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

checkpoint_e7 = ModelCheckpoint(
    "E7_VGG16_TL_OS_DA_best.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)


In [ ]:
history_e7 = model_e7.fit(
    train_generator_e7,
    validation_data=val_generator_e7,
    epochs=5,
    callbacks=[early_stopping_e7, checkpoint_e7],
    verbose=1
)


## Résultats de validation

Les cellules suivantes produisent les mêmes éléments de contrôle que pour les autres expériences : historique, courbes, matrice de confusion et rapport de classification. Elles utilisent **uniquement la validation**.


In [ ]:
import pandas as pd

history_e7_df = pd.DataFrame(history_e7.history)
history_e7_df.index = range(1, len(history_e7_df) + 1)
history_e7_df.index.name = "epoch"
display(history_e7_df)

history_e7_df.to_csv("E7_history.csv")


In [ ]:
import matplotlib.pyplot as plt

for metric in ["loss", "accuracy", "precision", "recall", "auc"]:
    plt.figure(figsize=(7, 4))
    plt.plot(history_e7.history[metric], label=f"train_{metric}")
    plt.plot(history_e7.history[f"val_{metric}"], label=f"val_{metric}")
    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.title(f"E7 — {metric}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"E7_{metric}.png", dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

val_generator_e7.reset()
val_prob_e7 = model_e7.predict(val_generator_e7, verbose=1).ravel()
val_pred_e7 = (val_prob_e7 >= 0.5).astype(int)
y_true_e7 = val_generator_e7.classes

cm_e7 = confusion_matrix(y_true_e7, val_pred_e7)
print(cm_e7)
print()
print(classification_report(
    y_true_e7,
    val_pred_e7,
    target_names=["Non-COVID", "COVID"],
    digits=4
))

report_e7 = classification_report(
    y_true_e7,
    val_pred_e7,
    target_names=["Non-COVID", "COVID"],
    output_dict=True
)
pd.DataFrame(report_e7).transpose().to_csv("E7_classification_report.csv")

plt.figure(figsize=(5, 4))
plt.imshow(cm_e7, cmap="Blues")
plt.title("E7 — Matrice de confusion (validation)")
plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.xticks([0, 1], ["Non-COVID", "COVID"])
plt.yticks([0, 1], ["Non-COVID", "COVID"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm_e7[i, j], ha="center", va="center")

plt.tight_layout()
plt.savefig("E7_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
